# Home Cleaning Task Scheduler — Strands Agents + Amazon Bedrock

A starter notebook for scheduling and maintaining cleaning tasks for a maid/helper, including buckets, dustbins, fans, refrigerator, and patio.

**Features**
- Persists tasks and completion history in SQLite.
- Lists tasks due today, overdue, or upcoming.
- Uses a Strands agent backed by Amazon Bedrock to suggest a realistic schedule.
- Requires explicit Python function calls to mark tasks complete; the model does not silently change records.

**Setup:** configure AWS credentials and a Bedrock model available to your account/region. Never put AWS access keys directly in the notebook.


## 1. Install dependencies
Run this cell in Jupyter after placing `requirements.txt` beside the notebook.

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 2. Configure Amazon Bedrock
Set `AWS_REGION` and optionally `BEDROCK_MODEL_ID` in your environment before starting Jupyter. The default model ID is an example; use a model enabled in your AWS account and region.


In [2]:
import os
import boto3
from strands import Agent
from strands.models import BedrockModel

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "us.amazon.nova-lite-v1:0")

session = boto3.Session(region_name=AWS_REGION)
if session.get_credentials() is None:
    raise RuntimeError(
        "AWS credentials not found. Configure an AWS profile, environment credentials, "
        "or an IAM role before using Amazon Bedrock."
    )

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.2,
)
print(f"Configured Bedrock model: {MODEL_ID} in {AWS_REGION}")


Configured Bedrock model: us.amazon.nova-lite-v1:0 in us-east-1


In [13]:

GUARDRAIL_ID = "rqyeydzpz0tp"
GUARDRAIL_VERSION = "1"


## 3. Create the local task store
Tasks persist in `cleaning_tasks.db` in the notebook's working directory. Frequency is represented as a number of days.

In [3]:
import sqlite3
from datetime import date, timedelta
from pathlib import Path

DB_PATH = Path("cleaning_tasks.db")

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS tasks (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL UNIQUE,
                area TEXT NOT NULL,
                frequency_days INTEGER NOT NULL CHECK (frequency_days > 0),
                estimated_minutes INTEGER NOT NULL DEFAULT 10,
                notes TEXT NOT NULL DEFAULT '',
                last_completed TEXT
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS completions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                task_id INTEGER NOT NULL,
                completed_on TEXT NOT NULL,
                completed_by TEXT NOT NULL DEFAULT 'Maid/helper',
                notes TEXT NOT NULL DEFAULT '',
                FOREIGN KEY(task_id) REFERENCES tasks(id)
            )
        """)

def seed_tasks():
    sample_tasks = [
        ("Clean buckets", "Bathroom/utility", 7, 10, "Rinse, scrub, and air-dry buckets."),
        ("Clean dustbins", "Whole home", 3, 15, "Empty first; wash and dry bins; replace liners."),
        ("Clean fans", "Bedrooms/living room", 30, 30, "Dust blades carefully; switch power off first."),
        ("Clean refrigerator", "Kitchen", 14, 30, "Discard spoiled food with permission; wipe shelves and handles."),
        ("Clean patio", "Patio/balcony", 7, 25, "Sweep floor and wipe accessible surfaces."),
    ]
    with get_connection() as conn:
        for name, area, frequency, minutes, notes in sample_tasks:
            conn.execute(
                """INSERT OR IGNORE INTO tasks
                   (name, area, frequency_days, estimated_minutes, notes)
                   VALUES (?, ?, ?, ?, ?)""",
                (name, area, frequency, minutes, notes),
            )

init_db()
seed_tasks()
print(f"Task database ready: {DB_PATH.resolve()}")


Task database ready: /home/ec2-user/SageMaker/cleaning_tasks.db


## 4. Scheduling and maintenance helpers
A task is due when it has never been completed or its recurrence interval has elapsed.

In [4]:
def list_tasks():
    with get_connection() as conn:
        rows = conn.execute("SELECT * FROM tasks ORDER BY area, name").fetchall()
    return [dict(row) for row in rows]

def task_status(task, today=None):
    today = today or date.today()
    last = date.fromisoformat(task["last_completed"]) if task["last_completed"] else None
    due_date = last + timedelta(days=task["frequency_days"]) if last else today
    days_until_due = (due_date - today).days
    status = "overdue" if days_until_due < 0 else ("due today" if days_until_due == 0 else "upcoming")
    return {
        **task,
        "due_date": due_date.isoformat(),
        "days_until_due": days_until_due,
        "status": status,
    }

def get_schedule(today=None):
    today = today or date.today()
    items = [task_status(t, today) for t in list_tasks()]
    return sorted(items, key=lambda x: (x["days_until_due"], -x["estimated_minutes"]))

def show_schedule():
    for item in get_schedule():
        last = item["last_completed"] or "Never"
        print(
            f'[{item["status"].upper():9}] {item["name"]:<20} '
            f'due={item["due_date"]} | ~{item["estimated_minutes"]} min | last done: {last}'
        )

def mark_completed(task_name, completed_on=None, completed_by="Maid/helper", notes=""):
    completed_on = completed_on or date.today().isoformat()
    date.fromisoformat(completed_on)  # validate date
    with get_connection() as conn:
        task = conn.execute("SELECT id FROM tasks WHERE name = ?", (task_name,)).fetchone()
        if task is None:
            available = ", ".join(t["name"] for t in list_tasks())
            raise ValueError(f"Unknown task {task_name!r}. Available tasks: {available}")
        conn.execute(
            "INSERT INTO completions (task_id, completed_on, completed_by, notes) VALUES (?, ?, ?, ?)",
            (task["id"], completed_on, completed_by, notes),
        )
        conn.execute("UPDATE tasks SET last_completed = ? WHERE id = ?", (completed_on, task["id"]))
    return f"Recorded completion: {task_name} on {completed_on}."

def add_task(name, area, frequency_days, estimated_minutes=10, notes=""):
    if not name.strip() or not area.strip():
        raise ValueError("Task name and area are required.")
    if int(frequency_days) < 1 or int(estimated_minutes) < 1:
        raise ValueError("Frequency and estimated minutes must be positive.")
    with get_connection() as conn:
        conn.execute(
            """INSERT INTO tasks (name, area, frequency_days, estimated_minutes, notes)
               VALUES (?, ?, ?, ?, ?)""",
            (name.strip(), area.strip(), int(frequency_days), int(estimated_minutes), notes.strip()),
        )
    return f"Added task: {name}"

show_schedule()


[DUE TODAY] Clean fans           due=2026-10-08 | ~30 min | last done: Never
[DUE TODAY] Clean refrigerator   due=2026-10-08 | ~30 min | last done: Never
[DUE TODAY] Clean patio          due=2026-10-08 | ~25 min | last done: Never
[DUE TODAY] Clean dustbins       due=2026-10-08 | ~15 min | last done: Never
[DUE TODAY] Clean buckets        due=2026-10-08 | ~10 min | last done: Never


## 5. Strands cleaning assistant
The model receives the current task list and suggests a plan. It is read-only; use `mark_completed()` to update the database.

In [5]:
SYSTEM_PROMPT = """
You are a practical household cleaning coordinator.
Help schedule and maintain cleaning tasks for a maid/helper.
Use only task data provided in the current request; do not invent completion history.
Spread heavy tasks across different days and respect estimated minutes and recurrence intervals.
Mention overdue and never-completed tasks. Give concise, actionable checklists.
For fans, remind the user to switch off power before cleaning.
For refrigerator cleaning, follow the household's instructions for food disposal.
Never claim you updated the database or marked a task complete.
"""
guardrail_id=GUARDRAIL_ID,
guardrail_version=GUARDRAIL_VERSION
cleaning_agent = Agent(model=bedrock_model, system_prompt=SYSTEM_PROMPT)

def ask_cleaning_agent(question):
    task_lines = []
    for t in get_schedule():
        task_lines.append(
            f'- {t["name"]} | area: {t["area"]} | every {t["frequency_days"]} days '
            f'| estimate: {t["estimated_minutes"]} min | last completed: {t["last_completed"] or "never"} '
            f'| due: {t["due_date"]} | status: {t["status"]} | notes: {t["notes"]}'
        )
    prompt = (
        f"Today's date is {date.today().isoformat()}.\n"
        "Current cleaning task list:\n" + "\n".join(task_lines)
        + f"\n\nUser request: {question}\nCreate a useful response based on this data."
    )
    return cleaning_agent(prompt)

# Example (run after Bedrock access is configured):
# print(ask_cleaning_agent("Create a balanced weekly plan grouped by day."))


## 6. Record completed work
Uncomment and edit the example. Completion is recorded in both the task's latest date and a history table.

In [7]:
print(mark_completed("Clean dustbins", completed_by="Maid", notes="Kitchen and bathroom bins washed"))
show_schedule()


Recorded completion: Clean dustbins on 2026-10-08.
[DUE TODAY] Clean fans           due=2026-10-08 | ~30 min | last done: Never
[DUE TODAY] Clean refrigerator   due=2026-10-08 | ~30 min | last done: Never
[DUE TODAY] Clean patio          due=2026-10-08 | ~25 min | last done: Never
[DUE TODAY] Clean buckets        due=2026-10-08 | ~10 min | last done: Never
[UPCOMING ] Clean dustbins       due=2026-10-11 | ~15 min | last done: 2026-10-08


## 7. Optional interactive console
Use `schedule` to view due tasks, `complete` to record work, `ask` to query the Bedrock agent, or `quit` to exit.

In [15]:
def cleaning_console():
    print("Cleaning Task Assistant")
    print("Commands: schedule | complete | ask | quit")
    while True:
        command = input("\nCommand: ").strip().lower()
        if command in {"quit", "q", "exit"}:
            print("Goodbye!")
            break
        elif command == "schedule":
            show_schedule()
        elif command == "complete":
            print("Tasks:", ", ".join(t["name"] for t in list_tasks()))
            name = input("Task name exactly as listed: ").strip()
            notes = input("Completion notes (optional): ").strip()
            try:
                print(mark_completed(name, notes=notes))
            except Exception as exc:
                print(f"Could not record completion: {exc}")
        elif command == "ask":
            question = input("What would you like help planning? ").strip()
            try:
                print(ask_cleaning_agent(question))
            except Exception as exc:
                print(f"Agent request failed: {exc}")
                print("Check AWS credentials, region, Bedrock model access, and model ID.")
        else:
            print("Unknown command. Choose schedule, complete, ask, or quit.")

#Uncomment to start:
cleaning_console()


Cleaning Task Assistant
Commands: schedule | complete | ask | quit



Command:  ask
What would you like help planning?  what bleach can i use to clean dustbin


Agent request failed: An error occurred (AccessDeniedException) when calling the ConverseStream operation: User: arn:aws:sts::020230974027:assumed-role/AmazonSageMaker-ExecutionRole-20260929T174891/SageMaker is not authorized to perform: bedrock:InvokeModelWithResponseStream on resource: arn:aws:bedrock:us-east-1:020230974027:inference-profile/us.amazon.nova-lite-v1:0 because no identity-based policy allows the bedrock:InvokeModelWithResponseStream action
└ Bedrock region: us-east-1
└ Model id: us.amazon.nova-lite-v1:0
Check AWS credentials, region, Bedrock model access, and model ID.



Command:  exit


Goodbye!


## 8. Next improvements
- Add preferred cleaning days and a maximum daily workload.
- Add household-specific instructions and a sign-off checklist.
- Add a Streamlit UI or calendar export.
- Back up `cleaning_tasks.db`.
- For multi-user use, move to a shared database and add authentication.

**Privacy:** task names, notes, and dates are sent to Amazon Bedrock when you call the agent. Avoid sensitive personal details.
